# Explore Data and Filter Out Non-PII

## Set parameters
Load in  raw data as a Spark Dataframe and set the name of the target column to be analyzed for PII.

In [ ]:
data_path = "catalog.schema.table"
data = spark.sql("""select * from """ + data_path)
target_col = "MY_TARGET"
output_table_name = "catalog.schema.output_table"
access_group = "group-name"

## Basic Exploration
Note that if using Databricks, you can use _display(df)_ instead of _df.show()_ for a more interactive and cleaner output.

In [ ]:
# show columns and data types
query = "DESCRIBE {df}"
spark.sql(query, df=data).show()

In [ ]:
# show row count
query = "SELECT COUNT(*) FROM {df}"
spark.sql(query, df=data).show()

In [ ]:
# show empty string target col row count
query = "SELECT COUNT(*) FROM {df} WHERE " + target_col + "=''"
spark.sql(query, df=data).show()

In [ ]:
# show null target col row count
query = "SELECT COUNT(*) FROM {df} WHERE " + target_col + " is null "
spark.sql(query, df=data).show()

In [ ]:
# see how many distinct values for target column
query = "SELECT COUNT(DISTINCT(" + target_col + ")) FROM {df}"
spark.sql(query, df=data).show()

In [ ]:
# See 100 most common values for target col
# Inspect visually for common non-PII patterns
query = (
    "SELECT "
    + target_col
    + ", COUNT(*) as count FROM {df} GROUP BY "
    + target_col
    + " ORDER BY count desc"
)
spark.sql(query, df=data).show(100)

## Round 1: Remove Patterns that are Unlikely to Have PII

In [ ]:
# List of non-PII regex patterns (case insensitive)
# Adjust for your use case if not analyzing lab test results
# Escaped regex characters to work with Python string formatting

non_pii_patterns = [
    r"^-?[0-9]+([.][0-9]+)?$",  # Number
    r"^[0-9]{{2}}/[0-9]{{2}}/[0-9]{{4}}$",  # Date (MM/DD/YYYY)
    r"^[0-9]{{2}}/[0-9]{{2}}/[0-9]{{2}}$",  # Date (MM/DD/YY)
    r"^-?[0-9]+([.][0-9]+)?[ ]*?%$",  # Number + %
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?x10e3/uL$",  # Number + x10e3/uL
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?mg/dL$",  # Number + mg/dL
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?g/dL$",  # Number + g/dL
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?ng/mL$",  # Number + ng/mL
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?mmol/L$",  # Number + mmol/L
    r"^[A-TV-Z][0-9][0-9AB]\.?[0-9A-TV-Z]{{0,4}}$",  # ICD-10 code
]

In [ ]:
# Show how many rows match each pattern
for pattern in non_pii_patterns:
    query = (
        "select count(*) from {df} where "
        + target_col
        + " rlike '"
        + pattern
        + "'"
    )
    print(query)
    counts_df = spark.sql(query, df=data)
    counts_df.show()

In [ ]:
# Non regex filter - filter out 1 and 2 character records
query_filter_1 = "SELECT * FROM {df} WHERE length(" + target_col + ")>2 "

# Assemble query to filter out these non-PII patterns
regex_query = ""
for pattern in non_pii_patterns:
    condition = "AND " + target_col + " not rlike '" + pattern + "' "
    regex_query += condition


query_filter_1 += regex_query
print(query_filter_1)

In [ ]:
# Apply 1st set of filters
data_filtered_1 = spark.sql(query_filter_1, df=data)
data_filtered_1.select(target_col).show()

In [ ]:
# How many rows left after 1st filter
query = "SELECT COUNT(*) FROM {df}"
spark.sql(query, df=data_filtered_1).show()

## Round 2: Remove Additional Patterns that are Unlikely to Have PII

In [ ]:
# Inspect non-PII patterns in data left after filter 1
# See 100 most common values in target col
# Inspect visually for additional common non-PII patterns

query = (
    "SELECT "
    + target_col
    + ", COUNT(*) as count FROM {df} GROUP BY "
    + target_col
    + " ORDER BY count desc"
)
spark.sql(query, df=data_filtered_1).show(100)

In [ ]:
# List of non-PII regex patterns (case insensitive)
# Adjust for your use case if not analyzing lab test results
# Escaped regex characters to work with Python string formatting

non_pii_patterns = [
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?IU/L$",  # Number + IU/L
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?U/L$",  # Number + U/L
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?fL$",  # Number + fL
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?mL/min/1\.73$",  # Number + mL/min/1.73 (literal dot)
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?pg$",  # Number + pg
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?(calc)$",  # Number + calc
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?\\(calc\\)$",  # Number + (calc)
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?K/uL$",  # Number + K/uL
    r"(?i)^-?[0-9]+([.][0-9]+)?[ ]*?IU/mL$",  # Number + IU/mL
    r"^>[ ]*-?[0-9]+([.][0-9]+)?$",  # > number
    r"^<[ ]*-?[0-9]+([.][0-9]+)?$",  # < number
]

In [ ]:
# Show how many rows match each pattern

for pattern in non_pii_patterns:
    query = (
        "select count(*) from {df} where "
        + target_col
        + " rlike '"
        + pattern
        + "'"
    )
    print(query)
    counts_df = spark.sql(query, df=data_filtered_1)
    counts_df.show()

In [ ]:
# Assemble query to filter out these non-PII patterns

query_filter_2 = "SELECT * FROM {df} WHERE "

regex_query = ""
for pattern in non_pii_patterns:
    condition = "AND " + target_col + " not rlike '" + pattern + "' "
    regex_query += condition
# Remove extra AND
regex_query = regex_query[4:]

query_filter_2 += regex_query
print(query_filter_2)

In [ ]:
# Apply 2nd set of filters
data_filtered_2 = spark.sql(query_filter_2, df=data_filtered_1)
data_filtered_2.select(target_col).show()

In [ ]:
# How many rows left after 2nd filter
query = "SELECT COUNT(*) FROM {df}"
spark.sql(query, df=data_filtered_2).show()

## Round 3: Remove Common Terms that are Non-PII

In [ ]:
# Inspect data left after filter 2
# See 1000 most common values for target col

query = (
    "SELECT "
    + target_col
    + ", COUNT(*) as count FROM {df} GROUP BY "
    + target_col
    + " ORDER BY count desc"
)
spark.sql(query, df=data_filtered_2).show(1000)

In [ ]:
# If all 1000 terms look PII free, filter them out of the dataset as well.
# If there is some PII in the list, you can create the top 1000 list and then drop out rows that have PII.

query = (
    "SELECT "
    + target_col
    + ", COUNT(*) as count FROM {df} GROUP BY "
    + target_col
    + " ORDER BY count desc LIMIT 1000"
)
top_1000_terms = spark.sql(query, df=data_filtered_2)

In [ ]:
# Apply 3rd set of filters

query_filter_3 = (
    "SELECT * FROM {df} WHERE "
    + target_col
    + " NOT IN (SELECT "
    + target_col
    + " FROM {top_terms_df} )"
    ""
)
data_filtered_3 = spark.sql(
    query_filter_3, df=data_filtered_2, top_terms_df=top_1000_terms
)

In [ ]:
# How many rows left after 3rd  filter
query = "SELECT COUNT(*) FROM {df}"
spark.sql(query, df=data_filtered_3).show()

Conduct more rounds of exploration and filtering as desired

## Final filtered table

In [ ]:
query = (
    "CREATE OR REPLACE TABLE " + output_table_name + " AS SELECT * FROM {df}"
)
spark.sql(query, df=data_filtered_3)

In [ ]:
spark.sql("GRANT SELECT ON " + output_table_name + " TO " + access_group)